In [7]:
import json, re
import numpy as np 
import pandas as pd

TRAIN = True

import nltk
import ssl
try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context
resources = ['wordnet', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng', 'sentiwordnet', 'omw-1.4']
for resource in resources:
    try: nltk.download(resource, quiet=True)
    except: pass

In [8]:
import os
import torch
from transformers.utils import is_torch_available
from load import parse_dataset

if TRAIN:
    dataset='../../datasets/slovene/train/SemEval2018-T3-train-taskA_emoji.txt'
    corpus, _ = parse_dataset(dataset)
    corpus_preprocessed = json.load(open('../../extra_resources/train_preprocessed.txt','r'))
else:
    dataset='../../datasets/slovene/test_TaskA/SemEval2018-T3_input_test_taskA_emoji.txt'
    corpus = parse_dataset(dataset)
    corpus_preprocessed = json.load(open('../../extra_resources/test_preprocessed.txt','r'))

In [9]:
# intensity features - 3 binarized features for splitted tweets which show:
# 1) the intensity of the left half
# 2) the intensity of the right half
# 3) the difference between the polarities of left and right halves

from functools import lru_cache

# To make the pipeline run smoothly without a Java server, we retain the 
# transformer sentiment logic to fake the 0-4 outputs from Stanford CoreNLP
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

def chunkIt(seq, n):
    avg = len(seq) / float(n)
    out = []
    last = 0.0
    while last < len(seq):
        out.append(seq[int(last):int(last + avg)])
        last += avg
    return out

@lru_cache(maxsize=1)
def get_sentiment_pipeline():
    try:
        from transformers import pipeline
        return pipeline("sentiment-analysis", model=MODEL_NAME, tokenizer=MODEL_NAME)
    except Exception as e:
        return None

sentiment_cache = {}

def mock_sentiment_score(text):
    if isinstance(text, list): text = ' '.join(text)
    text = (text or '').strip()
    if not text: return 2
    cache_key = text.lower()
    if cache_key in sentiment_cache: 
        return sentiment_cache[cache_key]

    nlp = get_sentiment_pipeline()
    if nlp is None: return 2

    try:
        pred = nlp(text[:512])[0]
        label = str(pred.get('label', '')).lower()
        conf = float(pred.get('score', 0.0))
        if label in {'label_0', 'negative'}:
            score = 0 if conf >= 0.85 else 1
        elif label in {'label_1', 'neutral'}:
            score = 2
        else:
            score = 4 if conf >= 0.85 else 3
    except Exception:
        score = 2

    sentiment_cache[cache_key] = score
    return score

feats_1 = []
for text in corpus:
    part1, part2 = chunkIt(text, 2)
    output1 = mock_sentiment_score(part1)
    output2 = mock_sentiment_score(part2)

    leftIntensity = rightIntensity = polarityDiff = 0
    if output1 in [0, 4]: leftIntensity = 1
    if output2 in [0, 4]: rightIntensity = 1
    if (output1 > 2 and output2 < 2) or (output1 < 2 and output2 > 2):
        polarityDiff = 1
    feats_1.append(np.array([leftIntensity, rightIntensity, polarityDiff]))

In [10]:
# contrast
df = pd.read_csv('../../extra_resources/Emoji_Sentiment_Data_v1.0.csv')
df = df[['Emoji', 'Negative', 'Neutral', 'Positive']]
tuples = [tuple(x) for x in df.values]

idx2lb = {0: -1, 1: 0, 2: 1}
emoji_sentimens = {}
for val in tuples:
    emoji_sentimens[val[0]] = idx2lb[np.argmax(np.array(val[1:]))]

def extractEmoticon(tweet):
    return re.findall(r'[\U0001f600-\U0001f650]', ' '.join(tweet))

twts = [extractEmoticon(twt[0]) for twt in corpus_preprocessed]
twts = [[emoji_sentimens[emoji] for emoji in twt if emoji in emoji_sentimens] for twt in twts]

def extractHashtag(tweet):
    t = tweet.split(' ')
    text = []
    hashtagText = []
    oneHashtag = []
    flag = 0
    for w in t:
        if w == "<hashtag>":
            flag = 1
            continue
        if flag == 1:
            if w == "</hashtag>":
                hashtagText.append(oneHashtag)
                oneHashtag = []
                flag = 0
            else:
                oneHashtag.append(w)
        else:
            text.append(w)
    return text, hashtagText

txt = [extractHashtag(tweet) for tweet in corpus_preprocessed]

assert len(txt) == len(twts)
txt = [(txt[i][0], txt[i][1], twts[i]) for i in range(len(twts))]

def sentiment(txt):
    txt_str = ' '.join(txt) if isinstance(txt, list) else txt
    if not len(txt_str): return 2
    return mock_sentiment_score(txt_str)

def contrast(twt):
    contrast = 0
    txt_sentiment = sentiment(twt[0])
    htag_sentiment = [sentiment(hash_segment) for hash_segment in twt[1]]
    emoji_sentiment = twt[2]

    hset = set(htag_sentiment)
    eset = set(emoji_sentiment)

    if (txt_sentiment in {2, 3, 4}) and (hset & {0, 1}): contrast = 1
    elif (txt_sentiment in {0, 1}) and (hset & {3, 4}): contrast = 1
    elif (txt_sentiment in {2, 3, 4}) and (-1 in eset): contrast = 1
    elif (txt_sentiment in {0, 1}) and (1 in eset): contrast = 1
    elif {-1, 1}.issubset(eset): contrast = 1
    elif {0, 4}.issubset(hset) or {0, 3}.issubset(hset) or {1, 4}.issubset(hset): contrast = 1
    elif (hset & {0, 1}) and (1 in eset): contrast = 1
    elif (hset & {3, 4}) and (-1 in eset): contrast = 1
    return contrast

contrast_feats = [np.array([contrast(twt)]) for twt in txt]

In [11]:
# ekphrasis-based features (extracted from pre-processed data)

tags =  ['<allcaps>', '<annoyed>', '<censored>', '<date>', '<elongated>', '<emphasis>', '<happy>',
         '<hashtag>', '<heart>', '<kiss>', '<laugh>', '<money>', '<number>', '<percent>', '<phone>',
         '<repeated>', '<sad>', '<shocking>', '<surprise>', '<time>', '<tong>', '<url>', '<user>',
         '<wink>']

def tweet_vecs(twt, n=2):
    twt = twt.split()
    chunks = chunkIt(twt, n)
    scores = []
    for chunk in chunks:
        for tag in tags:
            scores.append(sum(1 for t in chunk if t == tag))
    return scores

def feats(text):
    return [tweet_vecs(twt) for twt in text]

ekphrasis_feats = [np.array(v) for v in feats(corpus_preprocessed)]

from ekphrasis.utils.nlp import polarity
polarity_flag = True

polarity_vectors = []
for tweet in corpus_preprocessed:
    chunks = chunkIt(tweet, 2)
    polarity_vectors.append(np.concatenate(((polarity(chunks[0])[1], polarity(chunks[1])[1])), axis=0))

assert len(ekphrasis_feats) == len(polarity_vectors)

if polarity_flag: 
    ekphrasis_feats = [np.concatenate((ekphrasis_feats[i], polarity_vectors[i])) for i in range(len(ekphrasis_feats))]

In [12]:
min_len = min(len(feats_1), len(contrast_feats), len(ekphrasis_feats))
feats_1 = feats_1[:min_len]
contrast_feats = contrast_feats[:min_len]
ekphrasis_feats = ekphrasis_feats[:min_len]

# concatenate all the features (exactly 58 dimensions mapping to the English OG file)
features = np.concatenate((feats_1, contrast_feats, ekphrasis_feats), axis=1)
print("Final features shape:", features.shape)

# save the features in a numpy file 
if TRAIN:
    np.save('train_feats_taskA_og.npy', features)
else:
    np.save('test_feats_taskA_og.npy', features)

Final features shape: (2566, 58)
